In [3]:
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    classification_report,
    accuracy_score
)

import xgboost as xgb

import joblib
import os

In [4]:
# HEART DATASET


Xh_train = np.load("../data/processed/Xh_train.npy")
Xh_test = np.load("../data/processed/Xh_test.npy")

yh_train = np.load("../data/processed/yh_train.npy")
yh_test = np.load("../data/processed/yh_test.npy")


# DIABETES DATASET

Xd_train = np.load("../data/processed/Xd_train.npy")
Xd_test = np.load("../data/processed/Xd_test.npy")

yd_train = np.load("../data/processed/yd_train.npy")
yd_test = np.load("../data/processed/yd_test.npy")


# CANCER DATASET

Xc_train = np.load("../data/processed/Xc_train.npy")
Xc_test = np.load("../data/processed/Xc_test.npy")

yc_train = np.load("../data/processed/yc_train.npy")
yc_test = np.load("../data/processed/yc_test.npy")

print("All datasets loaded successfully!")

All datasets loaded successfully!


Reusable tuning function

In [5]:
def tune_and_evaluate(
    model,
    params,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name,
    dataset_name,
    search_type="grid"
):

    print(f"\n{'='*60}")
    print(f"Tuning {model_name} for {dataset_name}")
    print(f"{'='*60}")

    # Grid Search
    if search_type == "grid":

        search = GridSearchCV(
            estimator=model,
            param_grid=params,
            cv=5,
            scoring="recall",
            n_jobs=-1
        )

    # Random Search
    else:

        search = RandomizedSearchCV(
            estimator=model,
            param_distributions=params,
            n_iter=10,
            cv=5,
            scoring="recall",
            random_state=42,
            n_jobs=-1
        )

    # Train
    search.fit(X_train, y_train)

    # Best model
    best_model = search.best_estimator_

    # Prediction
    pred = best_model.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, pred)

    print("\nBest Parameters:")
    print(search.best_params_)

    print("\nAccuracy:", acc)

    print("\nClassification Report:")
    print(classification_report(y_test, pred))

    # Overfitting Check
    train_score = best_model.score(X_train, y_train)
    test_score = best_model.score(X_test, y_test)

    print("\nTrain Score:", train_score)
    print("Test Score:", test_score)

    # Save Model
    os.makedirs("../models/trained_models", exist_ok=True)

    model_path = f"../models/trained_models/{dataset_name}_{model_name}.pkl"

    joblib.dump(best_model, model_path)

    print(f"\nModel saved: {model_path}")

    return best_model, acc

Define Parameters for each models


In [6]:
#Random Forest Parameters

rf_params = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10, 15],
    "min_samples_split": [2, 5]
}

In [7]:
#SVM Parameters
svm_params = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf"],
    "gamma": ["scale", "auto"]
}

In [8]:
#XGBoost Parameters
xgb_params = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.8, 1.0]
}

Tuning the heart datasets

In [9]:
#Random Forest 
heart_rf, heart_rf_acc = tune_and_evaluate(
    RandomForestClassifier(random_state=42),
    rf_params,
    Xh_train,
    yh_train,
    Xh_test,
    yh_test,
    "RandomForest",
    "heart",
    "grid"
)


Tuning RandomForest for heart

Best Parameters:
{'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 100}

Accuracy: 0.9853658536585366

Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.99       102
           1       1.00      0.97      0.99       103

    accuracy                           0.99       205
   macro avg       0.99      0.99      0.99       205
weighted avg       0.99      0.99      0.99       205


Train Score: 1.0
Test Score: 0.9853658536585366

Model saved: ../models/trained_models/heart_RandomForest.pkl


In [10]:
#SVM
heart_svm, heart_svm_acc = tune_and_evaluate(
    SVC(probability=True),
    svm_params,
    Xh_train,
    yh_train,
    Xh_test,
    yh_test,
    "SVM",
    "heart",
    "grid"
)


Tuning SVM for heart

Best Parameters:
{'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}

Accuracy: 0.975609756097561

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.98      0.98       102
           1       0.98      0.97      0.98       103

    accuracy                           0.98       205
   macro avg       0.98      0.98      0.98       205
weighted avg       0.98      0.98      0.98       205


Train Score: 0.9975609756097561
Test Score: 0.975609756097561

Model saved: ../models/trained_models/heart_SVM.pkl


In [11]:
#XGBoost 
heart_xgb, heart_xgb_acc = tune_and_evaluate(
    xgb.XGBClassifier(
        eval_metric="logloss",
        random_state=42
    ),
    xgb_params,
    Xh_train,
    yh_train,
    Xh_test,
    yh_test,
    "XGBoost",
    "heart",
    "random"
)


Tuning XGBoost for heart

Best Parameters:
{'subsample': 1.0, 'n_estimators': 200, 'max_depth': 7, 'learning_rate': 0.1}

Accuracy: 0.9853658536585366

Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.99       102
           1       1.00      0.97      0.99       103

    accuracy                           0.99       205
   macro avg       0.99      0.99      0.99       205
weighted avg       0.99      0.99      0.99       205


Train Score: 1.0
Test Score: 0.9853658536585366

Model saved: ../models/trained_models/heart_XGBoost.pkl


Tuning diabetes models

In [12]:
#Random Forest
diabetes_rf, diabetes_rf_acc = tune_and_evaluate(
    RandomForestClassifier(random_state=42),
    rf_params,
    Xd_train,
    yd_train,
    Xd_test,
    yd_test,
    "RandomForest",
    "diabetes",
    "grid"
)


Tuning RandomForest for diabetes

Best Parameters:
{'max_depth': 15, 'min_samples_split': 2, 'n_estimators': 100}

Accuracy: 0.7597402597402597

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.79      0.81        99
           1       0.65      0.71      0.68        55

    accuracy                           0.76       154
   macro avg       0.74      0.75      0.74       154
weighted avg       0.77      0.76      0.76       154


Train Score: 1.0
Test Score: 0.7597402597402597

Model saved: ../models/trained_models/diabetes_RandomForest.pkl


In [13]:
#SVM 
diabetes_svm, diabetes_svm_acc = tune_and_evaluate(
    SVC(probability=True),
    svm_params,
    Xd_train,
    yd_train,
    Xd_test,
    yd_test,
    "SVM",
    "diabetes",
    "grid"
)


Tuning SVM for diabetes

Best Parameters:
{'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}

Accuracy: 0.6883116883116883

Classification Report:
              precision    recall  f1-score   support

           0       0.74      0.79      0.76        99
           1       0.57      0.51      0.54        55

    accuracy                           0.69       154
   macro avg       0.66      0.65      0.65       154
weighted avg       0.68      0.69      0.68       154


Train Score: 0.8843648208469055
Test Score: 0.6883116883116883

Model saved: ../models/trained_models/diabetes_SVM.pkl


In [14]:
#XGBoost 
diabetes_xgb, diabetes_xgb_acc = tune_and_evaluate(
    xgb.XGBClassifier(
        eval_metric="logloss",
        random_state=42
    ),
    xgb_params,
    Xd_train,
    yd_train,
    Xd_test,
    yd_test,
    "XGBoost",
    "diabetes",
    "random"
)


Tuning XGBoost for diabetes

Best Parameters:
{'subsample': 0.8, 'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.05}

Accuracy: 0.7467532467532467

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.76      0.79        99
           1       0.62      0.73      0.67        55

    accuracy                           0.75       154
   macro avg       0.73      0.74      0.73       154
weighted avg       0.76      0.75      0.75       154


Train Score: 0.9478827361563518
Test Score: 0.7467532467532467

Model saved: ../models/trained_models/diabetes_XGBoost.pkl


Tuning Cancer Models


In [15]:
#Random Forest 
cancer_rf, cancer_rf_acc = tune_and_evaluate(
    RandomForestClassifier(random_state=42),
    rf_params,
    Xc_train,
    yc_train,
    Xc_test,
    yc_test,
    "RandomForest",
    "cancer",
    "grid"
)


Tuning RandomForest for cancer

Best Parameters:
{'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 100}

Accuracy: 0.9649122807017544

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.93      0.95        43
           1       0.96      0.99      0.97        71

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114


Train Score: 0.9934065934065934
Test Score: 0.9649122807017544

Model saved: ../models/trained_models/cancer_RandomForest.pkl


In [16]:
#SVM
cancer_svm, cancer_svm_acc = tune_and_evaluate(
    SVC(probability=True),
    svm_params,
    Xc_train,
    yc_train,
    Xc_test,
    yc_test,
    "SVM",
    "cancer",
    "grid"
)


Tuning SVM for cancer

Best Parameters:
{'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'}

Accuracy: 0.9824561403508771

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.95      0.98        43
           1       0.97      1.00      0.99        71

    accuracy                           0.98       114
   macro avg       0.99      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114


Train Score: 0.9824175824175824
Test Score: 0.9824561403508771

Model saved: ../models/trained_models/cancer_SVM.pkl


In [17]:
#XGBoost 
cancer_xgb, cancer_xgb_acc = tune_and_evaluate(
    xgb.XGBClassifier(
        eval_metric="logloss",
        random_state=42
    ),
    xgb_params,
    Xc_train,
    yc_train,
    Xc_test,
    yc_test,
    "XGBoost",
    "cancer",
    "random"
)


Tuning XGBoost for cancer

Best Parameters:
{'subsample': 0.8, 'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.1}

Accuracy: 0.956140350877193

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.93      0.94        43
           1       0.96      0.97      0.97        71

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114


Train Score: 1.0
Test Score: 0.956140350877193

Model saved: ../models/trained_models/cancer_XGBoost.pkl


Final Comparison Table

In [18]:
results = pd.DataFrame({

    "Dataset": [
        "Heart",
        "Heart",
        "Heart",
        "Diabetes",
        "Diabetes",
        "Diabetes",
        "Cancer",
        "Cancer",
        "Cancer"
    ],

    "Model": [
        "RF",
        "SVM",
        "XGB",
        "RF",
        "SVM",
        "XGB",
        "RF",
        "SVM",
        "XGB"
    ],

    "Accuracy": [
        heart_rf_acc,
        heart_svm_acc,
        heart_xgb_acc,

        diabetes_rf_acc,
        diabetes_svm_acc,
        diabetes_xgb_acc,

        cancer_rf_acc,
        cancer_svm_acc,
        cancer_xgb_acc
    ]
})

results

,Dataset,Model,Accuracy
0,Heart,RF,0.985366
1,Heart,SVM,0.975610
2,Heart,XGB,0.985366
3,Diabetes,RF,0.759740
4,Diabetes,SVM,0.688312
5,Diabetes,XGB,0.746753
6,Cancer,RF,0.964912
7,Cancer,SVM,0.982456
8,Cancer,XGB,0.956140
